# 01 — Knowledge Tracing Data EDA

Phân tích dữ liệu canonical bằng pandas, chỉ hiển thị bảng và số liệu (không tạo biểu đồ).
Nguồn dữ liệu: `outputs/assistments_2009/canonical.csv`.

In [5]:
from ast import literal_eval
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

workspace_candidates = (Path.cwd(), *Path.cwd().parents)
WORKSPACE_ROOT = next(
    (
        candidate
        for candidate in workspace_candidates
        if (candidate / "scripts" / "adapt_assistments_2009.py").is_file()
    ),
    Path.cwd(),
)
DATA_PATH = WORKSPACE_ROOT / "outputs/assistments_2009/canonical.csv"
RAW_DATA_PATH = WORKSPACE_ROOT / "data/raw/assistments_2009.csv"
REQUIRED_COLUMNS = [
    "student_id",
    "interaction_id",
    "step",
    "question_id",
    "kc_ids",
    "correct",
]

if not DATA_PATH.is_file():
    if not RAW_DATA_PATH.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy dữ liệu canonical: {DATA_PATH}. "
            f"Hãy đặt dữ liệu raw tại {RAW_DATA_PATH} hoặc tạo canonical.csv trước."
        )
    sys.path.insert(0, str(WORKSPACE_ROOT))
    from scripts.adapt_assistments_2009 import adapt_file

    audit = adapt_file(RAW_DATA_PATH, DATA_PATH)
    print(f"Đã tạo dữ liệu canonical: {DATA_PATH}")
    print(f"Số dòng canonical: {audit['output_rows']:,}")

raw_df = pd.read_csv(DATA_PATH)
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(raw_df.columns))
if missing_columns:
    raise ValueError(f"Thiếu cột canonical bắt buộc: {missing_columns}")

# Chỉ giữ các dòng đủ trường bắt buộc và có correct là 0 hoặc 1.
df = raw_df.copy()
df["correct"] = pd.to_numeric(df["correct"], errors="coerce")
valid_mask = df[REQUIRED_COLUMNS].notna().all(axis=1) & df["correct"].isin([0, 1])
df = df.loc[valid_mask].copy()
if df.empty:
    raise ValueError("Không có lượt tương tác hợp lệ để phân tích.")


def parse_kc_ids(value):
    if isinstance(value, str):
        try:
            parsed = literal_eval(value)
        except (SyntaxError, ValueError):
            if value.lstrip().startswith("["):
                raise ValueError(f"kc_ids không phải list hợp lệ: {value!r}")
            parsed = value
    else:
        parsed = value

    if isinstance(parsed, (int, str)):
        parsed = [parsed]
    if not isinstance(parsed, list):
        raise ValueError(f"kc_ids phải là list hoặc một mã KC đơn, nhận được: {value!r}")
    return parsed


df["kc_ids_parsed"] = df["kc_ids"].map(parse_kc_ids)
print(f"Số dòng ban đầu: {len(raw_df):,}")
print(f"Số lượt tương tác hợp lệ: {len(df):,}")
print(f"Số dòng không hợp lệ đã loại: {len(raw_df) - len(df):,}")

Số dòng ban đầu: 283,105
Số lượt tương tác hợp lệ: 283,105
Số dòng không hợp lệ đã loại: 0


In [6]:
# Tổng quan dữ liệu và độ dài chuỗi tương tác theo học sinh.

all_kc_ids = {kc_id for kc_list in df["kc_ids_parsed"] for kc_id in kc_list}
student_interaction_counts = df.groupby("student_id").size()

overview = pd.DataFrame(
    {
        "Chỉ số": [
            "Tổng lượt tương tác hợp lệ",
            "Số học sinh duy nhất",
            "Số câu hỏi duy nhất",
            "Số kỹ năng/khái niệm duy nhất",
        ],
        "Giá trị": [
            len(df),
            df["student_id"].nunique(),
            df["question_id"].nunique(),
            len(all_kc_ids),
        ],
    }
)
display(overview)

display(
    pd.DataFrame(
        {
            "Số lượt tương tác mỗi học sinh": [
                student_interaction_counts.mean(),
                student_interaction_counts.min(),
                student_interaction_counts.max(),
            ]
        },
        index=["Trung bình", "Nhỏ nhất", "Lớn nhất"],
    )
)

low_interaction_stats = pd.DataFrame(
    {
        "Ngưỡng": ["Dưới 5 lượt", "Dưới 10 lượt"],
        "Số học sinh": [
            int((student_interaction_counts < 5).sum()),
            int((student_interaction_counts < 10).sum()),
        ],
        "Tỷ lệ (%)": [
            (student_interaction_counts < 5).mean() * 100,
            (student_interaction_counts < 10).mean() * 100,
        ],
    }
)
display(low_interaction_stats.round({"Tỷ lệ (%)": 2}))

student_question_counts = df.groupby(["student_id", "question_id"]).size()
repeated_students = (
    student_question_counts[student_question_counts > 1]
    .index.get_level_values("student_id")
    .nunique()
)
print(f"Số học sinh có ít nhất một câu hỏi được làm lặp lại: {repeated_students:,}")

,Chỉ số,Giá trị
0,Tổng lượt tương tác hợp lệ,283105
1,Số học sinh duy nhất,4163
2,Số câu hỏi duy nhất,17751
3,Số kỹ năng/khái niệm duy nhất,149


,Số lượt tương tác mỗi học sinh
Trung bình,68.005044
Nhỏ nhất,1.000000
Lớn nhất,1061.000000


,Ngưỡng,Số học sinh,Tỷ lệ (%)
0,Dưới 5 lượt,519,12.47
1,Dưới 10 lượt,1161,27.89


Số học sinh có ít nhất một câu hỏi được làm lặp lại: 608


In [7]:
# Tỷ lệ đúng toàn bộ và tỷ lệ đúng của từng học sinh.

overall_accuracy = df["correct"].mean()
student_accuracy = (
    df.groupby("student_id")["correct"]
    .mean()
    .rename("accuracy_rate")
    .to_frame()
)
print(f"Tỷ lệ trả lời đúng toàn bộ dữ liệu: {overall_accuracy:.2%}")
display(student_accuracy)

student_accuracy["Nhóm học lực"] = pd.cut(
    student_accuracy["accuracy_rate"],
    bins=[float("-inf"), 0.5, 0.8, float("inf")],
    labels=["Kém", "Khá/Trung bình", "Giỏi"],
    right=False,
)
mastery_groups = (
    student_accuracy["Nhóm học lực"]
    .value_counts()
    .reindex(["Giỏi", "Khá/Trung bình", "Kém"], fill_value=0)
    .rename("Số học sinh")
    .to_frame()
)
mastery_groups["Tỷ lệ (%)"] = mastery_groups["Số học sinh"] / len(student_accuracy) * 100
display(mastery_groups.round({"Tỷ lệ (%)": 2}))

Tỷ lệ trả lời đúng toàn bộ dữ liệu: 65.81%


,accuracy_rate
student_id,
0,0.315789
1,0.823529
2,0.833333
3,0.666667
4,0.705455
...,...
4158,0.822547
4159,0.844639
4160,0.826087


,Số học sinh,Tỷ lệ (%)
Nhóm học lực,,
Giỏi,1141,27.41
Khá/Trung bình,1976,47.47
Kém,1046,25.13


In [8]:
# EDA cho cấu hình task meta-learning (MAML)
# df là dataframe đã làm sạch; mỗi phần tử kc_ids_parsed là list KC của tương tác.

required_eda_columns = {"student_id", "kc_ids_parsed"}
missing_eda_columns = required_eda_columns - set(df.columns)
if missing_eda_columns:
    raise ValueError(f"df thiếu cột cần cho EDA MAML: {sorted(missing_eda_columns)}")

# 1. Kích thước task theo học sinh, hỗ trợ lựa chọn window size.
student_interaction_counts = df.groupby("student_id").size().rename("interaction_count")
student_task_summary = student_interaction_counts.describe(
    percentiles=[0.25, 0.50, 0.75]
).reindex(["min", "max", "mean", "50%", "25%", "75%"])
student_task_summary.index = ["min", "max", "mean", "median", "25%", "75%"]
display(student_task_summary.to_frame(name="Số lượt tương tác / học sinh").round(2))

# 2. Học sinh có từ 15 đến 100 lượt tương tác (bao gồm cả hai ngưỡng).
min_task_interactions, max_task_interactions = 15, 100
eligible_students = student_interaction_counts.between(
    min_task_interactions, max_task_interactions, inclusive="both"
)
student_count = len(student_interaction_counts)
eligible_count = int(eligible_students.sum())
eligible_rate = eligible_count / student_count * 100 if student_count else 0.0
print(
    f"Học sinh trong khoảng [{min_task_interactions}, {max_task_interactions}] lượt: "
    f"{eligible_count:,}/{student_count:,} ({eligible_rate:.2f}%)"
)

# 3. Số KC khác nhau mà mỗi học sinh đã tương tác.
exploded_student_skills = (
    df[["student_id", "kc_ids_parsed"]]
    .explode("kc_ids_parsed")
    .dropna(subset=["kc_ids_parsed"])
)
student_skill_counts = (
    exploded_student_skills.groupby("student_id")["kc_ids_parsed"]
    .nunique()
    .reindex(student_interaction_counts.index, fill_value=0)
    .rename("distinct_skill_count")
)
print(
    "Số kỹ năng khác nhau / học sinh (trung bình): "
    f"{student_skill_counts.mean():.2f}"
)
student_skill_summary = student_skill_counts.describe(
    percentiles=[0.25, 0.50, 0.75]
).reindex(["min", "max", "mean", "50%", "25%", "75%"])
student_skill_summary.index = ["min", "max", "mean", "median", "25%", "75%"]
display(student_skill_summary.to_frame(name="Số KC khác nhau / học sinh").round(2))

# 4. Phân bố lượt tương tác theo từng KC; tương tác đa KC được tính cho từng KC.
skill_interaction_counts = (
    exploded_student_skills.groupby("kc_ids_parsed")
    .size()
    .rename("interaction_count")
)
skill_task_summary = skill_interaction_counts.describe(
    percentiles=[0.25, 0.50, 0.75]
).reindex(["min", "max", "mean", "50%", "25%", "75%"])
skill_task_summary.index = ["min", "max", "mean", "median", "25%", "75%"]
display(skill_task_summary.to_frame(name="Số lượt tương tác / kỹ năng").round(2))

rare_skill_count = int((skill_interaction_counts < 50).sum())
popular_skill_count = int((skill_interaction_counts > 1000).sum())
print(f"Số kỹ năng Hiếm (< 50 lượt tương tác): {rare_skill_count:,}")
print(f"Số kỹ năng Phổ biến (> 1000 lượt tương tác): {popular_skill_count:,}")

,Số lượt tương tác / học sinh
min,1.00
max,1061.00
mean,68.01
median,20.00
25%,8.00
75%,60.50


Học sinh trong khoảng [15, 100] lượt: 1,803/4,163 (43.31%)
Số kỹ năng khác nhau / học sinh (trung bình): 10.33


,Số KC khác nhau / học sinh
min,1.00
max,109.00
mean,10.33
median,4.00
25%,2.00
75%,9.00


,Số lượt tương tác / kỹ năng
min,1.00
max,24253.00
mean,1900.03
median,906.00
25%,234.00
75%,1970.00


Số kỹ năng Hiếm (< 50 lượt tương tác): 21
Số kỹ năng Phổ biến (> 1000 lượt tương tác): 66


In [ ]:
# Mô phỏng MAML-KT: 5 tương tác Support + 25 tương tác Query mỗi học sinh.
# Thứ tự dòng trong df là thứ tự tương tác; hãy sắp xếp df theo thời gian trước nếu cần.
required_maml_columns = {"student_id", "kc_ids_parsed", "correct"}
missing_maml_columns = required_maml_columns - set(df.columns)
if missing_maml_columns:
    raise ValueError(f"df thiếu cột cần cho mô phỏng MAML-KT: {sorted(missing_maml_columns)}")

maml_df = df[["student_id", "kc_ids_parsed", "correct"]].copy()

def _to_skill_list(value):
    if isinstance(value, (list, tuple, set)):
        return list(value)
    if pd.isna(value):
        return []
    return [value]

maml_df["_skill_ids"] = maml_df["kc_ids_parsed"].apply(_to_skill_list)
student_interaction_counts = maml_df.groupby("student_id", sort=False).size()
eligible_student_ids = student_interaction_counts[
    student_interaction_counts >= 30
].index
eligible_maml_df = maml_df[maml_df["student_id"].isin(eligible_student_ids)]
print(f"Số học sinh đủ điều kiện (>= 30 tương tác): {len(eligible_student_ids):,}")

student_unseen_stats = []
unseen_query_interactions = 0
query_interaction_count = 0

for student_id, student_rows in eligible_maml_df.groupby("student_id", sort=False):
    support_rows = student_rows.iloc[:5]
    query_rows = student_rows.iloc[5:30]
    support_skill_ids = {
        skill_id
        for skill_ids in support_rows["_skill_ids"]
        for skill_id in skill_ids
    }
    query_skill_ids = [
        skill_id
        for skill_ids in query_rows["_skill_ids"]
        for skill_id in skill_ids
    ]
    unseen_skill_flags = [
        skill_id not in support_skill_ids for skill_id in query_skill_ids
    ]
    unseen_interaction_flags = [
        any(skill_id not in support_skill_ids for skill_id in skill_ids)
        for skill_ids in query_rows["_skill_ids"]
    ]
    student_unseen_stats.append(
        {
            "student_id": student_id,
            "unseen_skill_count": sum(unseen_skill_flags),
            "query_skill_count": len(query_skill_ids),
            "unseen_skill_rate": (
                sum(unseen_skill_flags) / len(query_skill_ids)
                if query_skill_ids
                else float("nan")
            ),
        }
    )
    unseen_query_interactions += sum(unseen_interaction_flags)
    query_interaction_count += len(query_rows)

maml_unseen_summary = pd.DataFrame(student_unseen_stats)
if maml_unseen_summary.empty:
    print("Không có học sinh nào đủ điều kiện để tạo task 5-shot Support / 25 Query.")
else:
    print(
        "Số skill occurrence Unseen trong Query / học sinh — "
        f"mean: {maml_unseen_summary['unseen_skill_count'].mean():.2f}, "
        f"median: {maml_unseen_summary['unseen_skill_count'].median():.2f}"
    )
    students_with_query_skills = maml_unseen_summary[
        maml_unseen_summary["query_skill_count"] > 0
    ]
    if students_with_query_skills.empty:
        print("Không có skill_id trong các Query set để tính tỷ lệ Unseen Skill.")
        extreme_student_count = 0
        extreme_student_rate = 0.0
    else:
        unseen_skill_rate = students_with_query_skills["unseen_skill_rate"]
        print(
            "Tỷ lệ skill occurrence Unseen trong Query / học sinh — "
            f"mean: {unseen_skill_rate.mean() * 100:.2f}%, "
            f"median: {unseen_skill_rate.median() * 100:.2f}%"
        )
        extreme_student_count = int((unseen_skill_rate == 1).sum())
        extreme_student_rate = extreme_student_count / len(unseen_skill_rate) * 100

    query_unseen_interaction_rate = (
        unseen_query_interactions / query_interaction_count * 100
        if query_interaction_count
        else 0.0
    )
    print(
        "Tỷ lệ tương tác Query có ít nhất một Unseen Skill: "
        f"{query_unseen_interaction_rate:.2f}%"
    )
    print(
        "Tỷ lệ học sinh có 100% skill trong Query là Unseen: "
        f"{extreme_student_rate:.2f}% "
        f"({extreme_student_count:,}/{len(students_with_query_skills):,})"
    )

In [ ]:
from collections.abc import Sequence

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset


class StudentTaskDataset(Dataset):
    """Mỗi học sinh là một task với các lượt đầu làm support/query."""

    def __init__(
        self,
        df: pd.DataFrame,
        min_len: int = 15,
        max_len: int = 100,
        k_support: int = 10,
        k_query: int = 5,
    ) -> None:
        required_columns = {"student_id", "kc_ids_parsed", "correct"}
        missing_columns = required_columns - set(df.columns)
        if missing_columns:
            raise ValueError(f"df thiếu các cột bắt buộc: {sorted(missing_columns)}")
        if min_len < 1 or max_len < min_len:
            raise ValueError("Cần 1 <= min_len <= max_len.")
        if k_support < 1 or k_query < 1:
            raise ValueError("k_support và k_query phải lớn hơn 0.")
        if k_support + k_query > min_len:
            raise ValueError("k_support + k_query không được lớn hơn min_len.")

        self.k_support = k_support
        self.k_query = k_query
        self.student_ids = []
        self.tasks = []

        # sort=False giữ thứ tự nhóm theo lần xuất hiện; thứ tự dòng trong nhóm được giữ nguyên.
        for student_id, student_df in df.groupby("student_id", sort=False):
            interaction_count = len(student_df)
            if not min_len <= interaction_count <= max_len:
                continue

            skill_ids = []
            for kc_list in student_df["kc_ids_parsed"]:
                if not isinstance(kc_list, (list, tuple, np.ndarray)) or len(kc_list) == 0:
                    raise ValueError(
                        f"kc_ids_parsed phải là list không rỗng (student_id={student_id!r})."
                    )
                skill_ids.append(kc_list[0])

            skill_ids = pd.to_numeric(pd.Series(skill_ids), errors="raise").to_numpy(dtype=np.int64)
            labels = pd.to_numeric(student_df["correct"], errors="raise").to_numpy(dtype=np.int64)
            if not np.isin(labels, [0, 1]).all():
                raise ValueError(f"correct chỉ được chứa 0 hoặc 1 (student_id={student_id!r}).")

            split_at = k_support
            query_end = split_at + k_query
            self.student_ids.append(student_id)
            self.tasks.append(
                (
                    torch.tensor(skill_ids[:split_at], dtype=torch.long),
                    torch.tensor(labels[:split_at], dtype=torch.float32),
                    torch.tensor(skill_ids[split_at:query_end], dtype=torch.long),
                    torch.tensor(labels[split_at:query_end], dtype=torch.float32),
                )
            )

    def __len__(self) -> int:
        return len(self.tasks)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.tasks[index]


def collate_maml_tasks(
    batch: Sequence[tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]],
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Gom từng phần của các task thành tensor có batch dimension đầu tiên."""
    if not batch:
        raise ValueError("Không thể gom một batch rỗng.")
    return tuple(torch.stack(items, dim=0) for items in zip(*batch))


def make_maml_dataloader(
    df: pd.DataFrame,
    batch_size: int,
    min_len: int = 15,
    max_len: int = 100,
    k_support: int = 10,
    k_query: int = 5,
    shuffle: bool = True,
    drop_last: bool = False,
) -> DataLoader:
    """Tạo DataLoader; batch_size là số task (học sinh) mỗi batch."""
    if batch_size < 1:
        raise ValueError("batch_size phải lớn hơn 0.")
    dataset = StudentTaskDataset(
        df,
        min_len=min_len,
        max_len=max_len,
        k_support=k_support,
        k_query=k_query,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        collate_fn=collate_maml_tasks,
    )


# Ví dụ: mỗi batch chứa 32 học sinh/task; tensor đầu ra có shape [32, 10] và [32, 5].
maml_loader = make_maml_dataloader(df, batch_size=32)
print(f"Số task (học sinh) hợp lệ: {len(maml_loader.dataset):,}")

## Ghi chú diễn giải

- Các dòng thiếu trường canonical bắt buộc hoặc có `correct` ngoài `{0, 1}` được loại khỏi phân tích; số dòng bị loại được in ở cell đầu.
- `kc_ids` được parse từ chuỗi biểu diễn list để đếm tập kỹ năng/khái niệm duy nhất.
- Nhóm học lực được tính từ accuracy của từng học sinh: Giỏi `>= 80%`, Khá/Trung bình `50%` đến dưới `80%`, Kém `< 50%`.
- Notebook chỉ tính toán và hiển thị số liệu/bảng bằng pandas; không tạo biểu đồ.